In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig
from src.research_validation import aligned_returns


# 01 Risk Free Data
Run after 01_Stock_Data. Download Treasury rates and the S&P 500 benchmark. The stock module's output supplies the required dates only; rates and benchmark are fetched online each time.


## 1. Date coverage


In [ ]:
cfg = ResearchConfig().validate()
train_prices = pd.read_parquet("train_prices.parquet")
test_prices = pd.read_parquet("test_prices.parquet")
start = (train_prices.index.min() - pd.Timedelta(days=14)).strftime("%Y-%m-%d")
end = (test_prices.index.max() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")


## 2. Download rates and benchmark
Yahoo ^IRX is the 13-week Treasury bill yield proxy, quoted in percent on a discount basis. Dividing by 100 gives an annual decimal yield, treated as an effective-rate proxy by the model. This is a yield proxy, not a realized Treasury investment return. The benchmark is Yahoo's ^GSPC price index, not a total-return index. Missing rate dates use only earlier available observations downstream.


In [ ]:
download = yf.download(
    ["^IRX", "^GSPC"], start=start, end=end,
    auto_adjust=False, progress=True, threads=False,
)
close = download["Close"]
risk_free_rates = close["^IRX"].dropna().sort_index() / 100
benchmark = close["^GSPC"].dropna().sort_index()
if risk_free_rates.empty or benchmark.empty:
    raise ValueError("Rate or benchmark download is empty. Retry before proceeding.")
if (risk_free_rates <= -1).any():
    raise ValueError("Annual decimal rates must be greater than -1.")
display(risk_free_rates.describe())
display(benchmark.head())


## 3. Verify coverage and save
Verify alignment before pair screening. Modules 09 and 12 use the same prior-session conversion of annual rates into daily rates.


In [ ]:
coverage = aligned_returns(
    pd.DataFrame({"equity": cfg.initial_capital}, index=test_prices.index),
    benchmark, risk_free_rates,
)
risk_free_rates.to_frame("risk_free_rate").to_parquet("risk_free_rates.parquet")
benchmark.to_frame("benchmark").to_parquet("benchmark_prices.parquet")
display(coverage[["market_return", "rf_daily"]].head())
risk_free_rates.plot(figsize=(10, 3), title="13-week Treasury annual yield proxy")
plt.show()
